# Prompt sensitivity analysis (v16)

This notebook tests whether the v16 main analysis findings are robust to
reasonable variation in the extraction prompt. It is a focused robustness
check — not a re-run of the main analysis.

## What this notebook does

1. Loads v16 outputs (paragraph chunks, cluster centroids derived from
   exemplars, framing lens mappings).
2. Draws a stratified random sample of paragraphs (~500).
3. Re-runs concern extraction on the sample under three prompt variants:
   - **V0** baseline — matches v16's exact prompt (sanity check that we can
     reproduce v16 on the sample).
   - **V1** "issues" — broader framing; treats concerns, trade-offs, and
     uncertainties as in scope.
   - **V3** "strict decontextualisation" — adds an instruction to return
     NO_CONCERN rather than invent a generic formulation if removing
     technology-specific language would change the meaning.
4. Computes paragraph-level diagnostics: coverage, density, semantic
   similarity to baseline.
5. **Pushes results through to the lens level**: for each variant, assigns
   each phrase to its nearest v16 cluster centroid, then aggregates to
   lens-level salience by technology, then computes AI-vs-non-AI lens
   distinctiveness. This is the test that matters for the paper.
6. Computes Adjusted Rand Index between baseline V0 and each variant on the
   subset of paragraphs where both produced ≥1 phrase.

## What this notebook does NOT do

- Does not re-cluster phrases from scratch under each variant. Instead it
  uses v16's cluster centroids as a fixed reference space, which isolates
  extraction sensitivity from clustering sensitivity. (Re-clustering would
  also work but conflates two questions.)
- Does not modify any v16 outputs.

## What you'll need

- The v16 outputs zip (specifically: `paragraph_chunks.csv`,
  `extracted_concerns.csv`, `cluster_exemplars.json`,
  `cluster_summary.csv`, `cluster_labels.json`,
  `framing_lens_mappings.json`, `concern_traceability_paragraphs.csv`).
- An OpenAI API key with budget for ~2,000 chat-completions calls and
  ~3,000 embedding calls (small — should cost well under $5).
- ~30 minutes of wall time end-to-end.

## What you'll get out

- A small CSV table per variant showing coverage, density, mean similarity
  to baseline, ARI to baseline.
- A lens-level distinctiveness table with AI minus non-AI shares under each
  variant — directly comparable to v16's `ai_distinctive_framings.csv`.
- A sample of disagreement examples for hand-inspection.

This is enough to support a paper Methods paragraph saying that the
headline AI-distinctiveness pattern is preserved across reasonable prompt
variants.


In [ ]:
# @title Install dependencies
!pip -q install openai pandas numpy scikit-learn matplotlib tqdm openpyxl

In [ ]:
# @title Imports & config
import os, json, re, time, math, random, zipfile, threading
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics.pairwise import cosine_similarity

# ---- run config ----
RANDOM_SEED = 42
SAMPLE_SIZE = 500          # number of paragraphs to draw for sensitivity
MIN_PER_TECHNOLOGY = 15    # floor per technology (smaller technologies still represented)
MAX_WORKERS = 6            # parallel API workers
LLM_MODEL = "gpt-5-mini"   # match v16
EMBEDDING_MODEL = "text-embedding-3-large"  # match v16

# ---- paths ----
RUN_ID = time.strftime("%Y%m%d_%H%M%S")
RUN_DIR = Path(f"/content/sensitivity_run_{RUN_ID}")
RUN_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = RUN_DIR / "cache"
CACHE_DIR.mkdir(exist_ok=True)
OUTPUT_DIR = RUN_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
print(f"Run directory: {RUN_DIR}")

In [ ]:
# @title Upload v16 outputs
# Two ways: (a) upload public_dialogue_analysis_outputs.zip from your v16 run,
# or (b) upload the individual CSV/JSON files. The cell auto-detects which.

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

V16_DIR = RUN_DIR / "v16_outputs"
V16_DIR.mkdir(exist_ok=True)

REQUIRED = [
    "paragraph_chunks.csv",
    "extracted_concerns.csv",
    "cluster_exemplars.json",
    "cluster_summary.csv",
    "cluster_labels.json",
    "framing_lens_mappings.json",
    "concern_traceability_paragraphs.csv",
]

if IN_COLAB:
    print("Upload either the v16 outputs zip OR the individual files listed above.")
    uploaded = files.upload()
    for name, content in uploaded.items():
        target = V16_DIR / name
        target.write_bytes(content)
        # If it's a zip, extract it
        if name.endswith(".zip"):
            with zipfile.ZipFile(target, "r") as zf:
                zf.extractall(V16_DIR)
            # Outputs may be at top level or in an "outputs/" subdir
else:
    print("Not in Colab. Place v16 outputs in:", V16_DIR)
    print("(Adjust the path below or upload manually.)")

# Find the directory containing the required files (top-level or in outputs/)
candidates = [V16_DIR, V16_DIR / "outputs"]
V16_DATA = None
for c in candidates:
    if c.exists() and all((c / f).exists() for f in REQUIRED[:3]):
        V16_DATA = c
        break

if V16_DATA is None:
    raise FileNotFoundError(
        f"Could not find v16 outputs in {V16_DIR}. "
        f"Need at least: {REQUIRED}"
    )

print(f"v16 outputs found at: {V16_DATA}")
missing = [f for f in REQUIRED if not (V16_DATA / f).exists()]
if missing:
    print(f"WARNING: missing files: {missing}")
else:
    print("All required v16 files present.")

In [ ]:
# @title API access
import getpass
from openai import OpenAI

if not os.environ.get("OPENAI_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    except Exception:
        os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

client = OpenAI()
print("Client ready.")

In [ ]:
# @title Load v16 paragraph chunks, traceability, and lens mappings

chunks_df = pd.read_csv(V16_DATA / "paragraph_chunks.csv")
print(f"Loaded {len(chunks_df)} paragraphs across {chunks_df['source_file'].nunique()} documents and {chunks_df['technology_meta'].nunique()} technologies.")

trace_df = pd.read_csv(V16_DATA / "concern_traceability_paragraphs.csv")
print(f"Loaded traceability for {len(trace_df)} paragraphs.")

# v16 cluster summary
cluster_summary = pd.read_csv(V16_DATA / "cluster_summary.csv")
print(f"Loaded summary of {len(cluster_summary)} clusters.")

# v16 cluster exemplars (used to derive centroids by re-embedding)
with open(V16_DATA / "cluster_exemplars.json") as f:
    cluster_exemplars = json.load(f)
print(f"Loaded exemplars for {len(cluster_exemplars)} clusters.")

# v16 framing lens mappings (cluster_id -> lens_name)
with open(V16_DATA / "framing_lens_mappings.json") as f:
    framing_lens_mappings = json.load(f)
# This is structured as {lens_name: {"cluster_ids": [...], ...}}; invert to cluster_id -> lens
cluster_to_lens = {}
for lens_name, info in framing_lens_mappings.items():
    for cid in info.get("cluster_ids", []):
        cluster_to_lens[int(cid)] = lens_name
print(f"Lens mappings cover {len(cluster_to_lens)} clusters across {len(framing_lens_mappings)} lenses.")

# v16 cluster labels (for nicer output)
with open(V16_DATA / "cluster_labels.json") as f:
    cluster_labels_dict = json.load(f)
CLUSTER_LABELS = {int(k): v.get("label", f"Cluster {k}") for k, v in cluster_labels_dict.items()}

In [ ]:
# @title Draw stratified random sample of paragraphs

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Stratify by technology with a per-technology floor
sample_pieces = []
for tech, group in chunks_df.groupby("technology_meta"):
    n_proportional = int(round(SAMPLE_SIZE * len(group) / len(chunks_df)))
    n_for_tech = max(MIN_PER_TECHNOLOGY, n_proportional)
    n_for_tech = min(n_for_tech, len(group))
    sample_pieces.append(group.sample(n=n_for_tech, random_state=RANDOM_SEED))

sample_df = pd.concat(sample_pieces, ignore_index=True)
# Trim if oversized
if len(sample_df) > SAMPLE_SIZE * 1.3:
    sample_df = sample_df.sample(int(SAMPLE_SIZE * 1.3), random_state=RANDOM_SEED).reset_index(drop=True)

print(f"Sample size: {len(sample_df)}")
print(f"\nBy technology:")
print(sample_df["technology_meta"].value_counts())

In [ ]:
# @title Prompt variants

# V0 matches v16's prompt EXACTLY. Useful as a sanity check that running on a
# subset reproduces v16-like results.
V0_SYSTEM = "Extract public concerns. Be concise. Remove technology-specific language."
V0_TEMPLATE = """Extract the core public concerns from this paragraph from a UK public dialogue report.

CRITICAL RULES:
1. Remove ALL technology-specific references (AI, nuclear, genetic, nano, etc.)
2. Extract the underlying concern that could apply to ANY technology
3. Keep phrases concise (3-10 words each)
4. Focus on what people are worried about, not factual statements

EXAMPLES:
- "People worried about AI making unfair decisions" → "unfair automated decisions"
- "Concerns about nuclear waste storage" → "long-term waste storage safety"
- "Distrust of government handling of genetic data" → "distrust of government data handling"
- "Fear that robots will take jobs" → "technology displacing employment"

Return 1-3 concern phrases, one per line. No bullets, no numbering.
If the paragraph contains no clear public concern, return "NO_CONCERN".

Paragraph:
{text}"""

# V1 broadens "concerns" to "issues" (concerns + trade-offs + uncertainties +
# governance questions). This tests whether the choice of analytic category
# matters: if the lens-level distinctiveness story holds under "issues", it's
# robust to the concerns-vs-issues framing.
V1_SYSTEM = "Extract public issues. Be concise. Remove technology-specific language."
V1_TEMPLATE = """Extract the key public issues from this paragraph from a UK public dialogue report.

CRITICAL RULES:
1. Focus on issues raised in the dialogue. These may include concerns, trade-offs,
   uncertainties, disagreements, or governance questions.
2. Remove ALL technology-specific references (AI, nuclear, genetic, nano, etc.)
3. Extract the underlying issue that could apply to ANY technology
4. Keep phrases concise (3-10 words each)
5. Do NOT extract purely factual statements or background description

EXAMPLES:
- "Debate about how AI decisions should be regulated" → "appropriate regulation of automated decisions"
- "Uncertainty about long-term effects of gene editing" → "uncertainty about long-term impacts"
- "Questions over who should oversee nuclear safety" → "responsibility for safety oversight"

Return 1-3 issue phrases, one per line. No bullets, no numbering.
If the paragraph contains no clear public issue, return "NO_CONCERN".

Paragraph:
{text}"""

# V3 tightens decontextualisation. It says: if you can't decontextualise without
# distorting meaning, return NO_CONCERN rather than invent a generic phrase. If
# the headline result holds even with this stricter filter, the conclusion is
# not driven by the LLM's tendency to over-generalise.
V3_SYSTEM = V0_SYSTEM
V3_TEMPLATE = """Extract the core public concerns from this paragraph from a UK public dialogue report.

CRITICAL RULES:
1. Remove ALL technology-specific references (AI, nuclear, genetic, nano, etc.)
2. Extract the underlying concern that could apply to ANY technology
3. Keep phrases concise (3-10 words each)
4. Focus on what people are worried about, not factual statements
5. If removing technology-specific language would substantially change or obscure
   the meaning, return "NO_CONCERN" rather than inventing a generic formulation.

Return 1-3 concern phrases, one per line. No bullets, no numbering.
If the paragraph contains no clear public concern under these rules, return "NO_CONCERN".

Paragraph:
{text}"""

PROMPTS = {
    "V0": {"name": "Concerns (baseline, matches v16)", "system": V0_SYSTEM, "template": V0_TEMPLATE},
    "V1": {"name": "Issues (broader framing)",          "system": V1_SYSTEM, "template": V1_TEMPLATE},
    "V3": {"name": "Concerns (strict decontextualisation)", "system": V3_SYSTEM, "template": V3_TEMPLATE},
}
print("Variants:", list(PROMPTS.keys()))

In [ ]:
# @title Decontextualisation filter (mirrors v16 exactly)

_TECH_TERM_PATTERN = re.compile(
    r"\b("
    r"ai|artificial intelligence|"
    r"nuclear|"
    r"genetic|genome|gene|crispr|embryo|stem cell|"
    r"nano|"
    r"robot|drone|autonomous|"
    r"quantum|"
    r"gm|"
    r"algorithm|algorithmic|automated|automation|"
    r"radiation|engineered|biometric|surveillance"
    r")\b",
    re.IGNORECASE,
)

def phrase_contains_tech_term(phrase: str) -> bool:
    return bool(_TECH_TERM_PATTERN.search(phrase or ""))

In [ ]:
# @title Run extraction for each variant
# Caching: per-variant cache file. Re-running is idempotent unless the
# sample changes.

_failures_by_variant = {vid: [] for vid in PROMPTS}
_failures_lock = threading.Lock()

def _record_failure(variant_id, chunk_id, reason):
    with _failures_lock:
        _failures_by_variant[variant_id].append((chunk_id, reason))

def extract_for_variant(variant_id, prompt_spec, row):
    chunk_id = row["chunk_id"]
    text = str(row["text"])[:2000]
    user_prompt = prompt_spec["template"].format(text=text)

    def _call():
        return client.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {"role": "system", "content": prompt_spec["system"]},
                {"role": "user", "content": user_prompt},
            ],
            max_completion_tokens=1500,
        )

    try:
        try:
            response = _call()
        except Exception:
            time.sleep(1.0)
            response = _call()  # one retry
        content = response.choices[0].message.content
        if content is None:
            _record_failure(variant_id, chunk_id, "empty_response")
            return chunk_id, []
        content = content.strip()
        if content == "NO_CONCERN" or not content:
            return chunk_id, []
        phrases = [p.strip() for p in content.split("\n") if p.strip() and p.strip() != "NO_CONCERN"]
        # apply same word-boundary filter as v16
        phrases = [p for p in phrases if not phrase_contains_tech_term(p)]
        return chunk_id, phrases
    except Exception as e:
        _record_failure(variant_id, chunk_id, f"exception: {type(e).__name__}: {e}")
        return chunk_id, []


# Run all variants
extractions = {}  # variant_id -> dict(chunk_id -> list of phrases)

for variant_id, prompt_spec in PROMPTS.items():
    cache_file = CACHE_DIR / f"extractions_{variant_id}.json"
    if cache_file.exists():
        with open(cache_file) as f:
            cached = json.load(f)
        # Validate cache covers our sample
        if all(cid in cached for cid in sample_df["chunk_id"].astype(str)):
            extractions[variant_id] = {cid: phrases for cid, phrases in cached.items() if cid in set(sample_df["chunk_id"].astype(str))}
            print(f"[{variant_id}] using cached ({len(extractions[variant_id])} paragraphs)")
            continue
        else:
            print(f"[{variant_id}] cache incomplete; regenerating")

    print(f"\n[{variant_id}] {prompt_spec['name']}: extracting from {len(sample_df)} paragraphs...")
    results = {}
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(extract_for_variant, variant_id, prompt_spec, row): row["chunk_id"]
                   for _, row in sample_df.iterrows()}
        for fut in tqdm(as_completed(futures), total=len(futures), desc=variant_id):
            cid, phrases = fut.result()
            results[str(cid)] = phrases

    extractions[variant_id] = results
    with open(cache_file, "w") as f:
        json.dump(results, f)
    n_fail = len(_failures_by_variant[variant_id])
    print(f"[{variant_id}] complete: {len(results)} paragraphs, {n_fail} failures")

# Failure summary
print("\n=== Extraction failure summary ===")
for vid, fails in _failures_by_variant.items():
    if fails:
        print(f"  {vid}: {len(fails)} failures")
        # Show top reasons
        from collections import Counter
        reason_counts = Counter(reason for _, reason in fails)
        for r, n in reason_counts.most_common(3):
            print(f"    {n}× {r[:100]}")
    else:
        print(f"  {vid}: 0 failures")

In [ ]:
# @title Paragraph-level diagnostics: coverage, density

diag_rows = []
for vid in PROMPTS:
    res = extractions[vid]
    n_paras = len(res)
    n_with_phrase = sum(1 for phrases in res.values() if phrases)
    coverage = n_with_phrase / n_paras if n_paras else 0
    total_phrases = sum(len(phrases) for phrases in res.values())
    density = total_phrases / n_paras if n_paras else 0
    avg_phrase_len = (
        np.mean([len(p.split()) for phrases in res.values() for p in phrases])
        if total_phrases else 0
    )
    diag_rows.append({
        "variant": vid,
        "name": PROMPTS[vid]["name"],
        "n_paragraphs": n_paras,
        "n_with_phrase": n_with_phrase,
        "coverage_rate": round(coverage, 3),
        "total_phrases": total_phrases,
        "phrases_per_paragraph": round(density, 3),
        "mean_phrase_words": round(avg_phrase_len, 2),
    })

diag_df = pd.DataFrame(diag_rows)
diag_df.to_csv(OUTPUT_DIR / "table1_paragraph_diagnostics.csv", index=False)
print(diag_df.to_string(index=False))

In [ ]:
# @title Derive v16 cluster centroids (by re-embedding exemplar phrases)
# We use exemplars rather than re-embedding the full v16 corpus because:
#   1. Exemplars are by definition the closest phrases to each cluster centroid
#      (computed in v16 by ranking cosine similarity to centroid).
#   2. ~600 embeddings is far cheaper than ~12,000.
#   3. The notebook should be self-contained: not require the v16 embedding
#      arrays to also be exported.

def embed_texts(texts, batch_size=256):
    """Embed a list of strings. Returns np.ndarray of shape (n, d)."""
    out = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        resp = client.embeddings.create(model=EMBEDDING_MODEL, input=batch)
        out.extend([d.embedding for d in resp.data])
    return np.array(out, dtype=np.float32)

# Collect exemplar phrases per cluster
exemplar_phrases_by_cluster = {}
for cid_str, info in cluster_exemplars.items():
    cid = int(cid_str)
    examples = info.get("exemplars", [])
    # exemplars list contains dicts with 'concern' (the phrase)
    phrases = [e["concern"] for e in examples if isinstance(e, dict) and "concern" in e]
    if phrases:
        exemplar_phrases_by_cluster[cid] = phrases

print(f"Will derive centroids for {len(exemplar_phrases_by_cluster)} clusters from "
      f"{sum(len(p) for p in exemplar_phrases_by_cluster.values())} exemplar phrases.")

# Cache the centroids so re-runs are free
centroids_cache = CACHE_DIR / "v16_centroids.npz"
if centroids_cache.exists():
    cached = np.load(centroids_cache)
    centroid_ids = cached["cluster_ids"]
    centroid_vectors = cached["centroids"]
    print(f"Loaded {len(centroid_ids)} centroids from cache.")
else:
    # Flatten all exemplars and remember their cluster IDs
    flat_phrases = []
    flat_cluster_ids = []
    for cid, phrases in exemplar_phrases_by_cluster.items():
        for p in phrases:
            flat_phrases.append(p)
            flat_cluster_ids.append(cid)

    print(f"Embedding {len(flat_phrases)} exemplar phrases...")
    flat_embeddings = embed_texts(flat_phrases)

    # L2-normalise and average per cluster
    norms = np.linalg.norm(flat_embeddings, axis=1, keepdims=True)
    flat_embeddings_normed = flat_embeddings / np.clip(norms, 1e-12, None)

    centroid_ids_list = []
    centroid_vectors_list = []
    for cid in sorted(exemplar_phrases_by_cluster.keys()):
        mask = np.array([fc == cid for fc in flat_cluster_ids])
        cluster_embeddings = flat_embeddings_normed[mask]
        centroid = cluster_embeddings.mean(axis=0)
        # L2-normalise the centroid
        centroid = centroid / max(np.linalg.norm(centroid), 1e-12)
        centroid_ids_list.append(cid)
        centroid_vectors_list.append(centroid)

    centroid_ids = np.array(centroid_ids_list)
    centroid_vectors = np.array(centroid_vectors_list, dtype=np.float32)

    np.savez(centroids_cache, cluster_ids=centroid_ids, centroids=centroid_vectors)
    print(f"Saved {len(centroid_ids)} centroids to cache.")

In [ ]:
# @title Embed variant phrases and assign each to nearest v16 cluster

def normalise_rows(arr):
    norms = np.linalg.norm(arr, axis=1, keepdims=True)
    return arr / np.clip(norms, 1e-12, None)

# Flatten all phrases across all variants for batch embedding
all_phrases = []
phrase_index = []  # list of (variant, chunk_id, phrase_idx_in_para)
for vid, res in extractions.items():
    for chunk_id, phrases in res.items():
        for j, phrase in enumerate(phrases):
            all_phrases.append(phrase)
            phrase_index.append((vid, chunk_id, j))

print(f"Embedding {len(all_phrases)} phrases across {len(extractions)} variants...")

embed_cache = CACHE_DIR / "variant_phrase_embeddings.npz"
if embed_cache.exists():
    _cached = np.load(embed_cache, allow_pickle=True)
    if _cached["n_phrases"] == len(all_phrases):
        all_embeddings = _cached["embeddings"]
        print("Loaded variant embeddings from cache.")
    else:
        all_embeddings = None
else:
    all_embeddings = None

if all_embeddings is None:
    all_embeddings = embed_texts(all_phrases)
    np.savez(embed_cache, embeddings=all_embeddings, n_phrases=len(all_phrases))
    print(f"Cached {len(all_phrases)} embeddings.")

all_embeddings_normed = normalise_rows(all_embeddings)

# Cosine similarity to each v16 centroid; argmax = assigned cluster
sims_to_centroids = all_embeddings_normed @ centroid_vectors.T  # (n_phrases, n_clusters)
assigned_cluster_idx = sims_to_centroids.argmax(axis=1)
max_similarity = sims_to_centroids.max(axis=1)
assigned_cluster_id = centroid_ids[assigned_cluster_idx]

# Build a long-form table: variant, chunk_id, phrase_idx, phrase, assigned_cluster_id, max_similarity
phrase_assign_rows = []
for (vid, chunk_id, j), phrase, cid, sim in zip(phrase_index, all_phrases, assigned_cluster_id, max_similarity):
    phrase_assign_rows.append({
        "variant": vid,
        "chunk_id": chunk_id,
        "phrase_idx": j,
        "phrase": phrase,
        "assigned_cluster_id": int(cid),
        "assigned_cluster_label": CLUSTER_LABELS.get(int(cid), f"Cluster {cid}"),
        "assigned_lens": cluster_to_lens.get(int(cid), "(unmapped)"),
        "max_similarity": float(sim),
    })

phrase_assign_df = pd.DataFrame(phrase_assign_rows)
phrase_assign_df.to_csv(OUTPUT_DIR / "phrase_assignments_by_variant.csv", index=False)
print(f"Wrote {len(phrase_assign_df)} phrase-level assignments.")
print(f"\nMean assignment similarity by variant:")
print(phrase_assign_df.groupby("variant")["max_similarity"].agg(["mean","median","count"]).round(3))

In [ ]:
# @title Lens-level distinctiveness by variant — THE MAIN COMPARISON

# Attach technology metadata to each phrase via chunk_id
chunk_meta = chunks_df[["chunk_id", "technology_meta"]].copy()
chunk_meta["chunk_id"] = chunk_meta["chunk_id"].astype(str)
phrase_assign_df["chunk_id"] = phrase_assign_df["chunk_id"].astype(str)

phrase_with_meta = phrase_assign_df.merge(chunk_meta, on="chunk_id", how="left")

# Drop unmapped clusters (clusters not in any framing lens)
phrase_with_meta = phrase_with_meta[phrase_with_meta["assigned_lens"] != "(unmapped)"]

# Compute lens shares per technology, per variant
lens_share_records = []
for vid in PROMPTS:
    sub = phrase_with_meta[phrase_with_meta["variant"] == vid]
    for tech, tech_group in sub.groupby("technology_meta"):
        tech_n = len(tech_group)
        if tech_n == 0:
            continue
        for lens, lens_group in tech_group.groupby("assigned_lens"):
            lens_share_records.append({
                "variant": vid,
                "technology": tech,
                "lens": lens,
                "lens_n": len(lens_group),
                "tech_n": tech_n,
                "lens_share": len(lens_group) / tech_n,
            })

lens_share_df = pd.DataFrame(lens_share_records)
lens_share_df.to_csv(OUTPUT_DIR / "lens_share_by_tech_by_variant.csv", index=False)

# AI minus non-AI lens shares per variant. Two baselines:
#   tech-weighted: each non-AI tech contributes equally
#   doc-weighted: pool all non-AI phrases
print("\n=== AI vs non-AI lens distinctiveness, by variant ===\n")

distinctiveness_records = []
for vid in PROMPTS:
    sub = phrase_with_meta[phrase_with_meta["variant"] == vid]
    if len(sub) == 0:
        continue

    # AI shares
    ai_sub = sub[sub["technology_meta"] == "AI"]
    ai_total = len(ai_sub)
    ai_lens_shares = (ai_sub.groupby("assigned_lens").size() / ai_total).to_dict() if ai_total else {}

    # Non-AI doc-weighted
    nonai_sub = sub[sub["technology_meta"] != "AI"]
    nonai_total = len(nonai_sub)
    nonai_doc_lens_shares = (nonai_sub.groupby("assigned_lens").size() / nonai_total).to_dict() if nonai_total else {}

    # Non-AI tech-weighted: average within-tech share across non-AI techs
    nonai_techs = [t for t in sub["technology_meta"].unique() if t != "AI"]
    tech_lens_share_matrix = {}
    for t in nonai_techs:
        t_sub = sub[sub["technology_meta"] == t]
        t_total = len(t_sub)
        if t_total == 0:
            continue
        for lens in framing_lens_mappings:
            lens_n = (t_sub["assigned_lens"] == lens).sum()
            tech_lens_share_matrix.setdefault(lens, []).append(lens_n / t_total)
    nonai_tech_lens_shares = {lens: float(np.mean(vals)) for lens, vals in tech_lens_share_matrix.items()}

    for lens in framing_lens_mappings:
        ai_share = ai_lens_shares.get(lens, 0.0)
        nonai_doc = nonai_doc_lens_shares.get(lens, 0.0)
        nonai_tech = nonai_tech_lens_shares.get(lens, 0.0)
        distinctiveness_records.append({
            "variant": vid,
            "lens": lens,
            "ai_share": round(ai_share, 4),
            "nonai_share_tech_weighted": round(nonai_tech, 4),
            "nonai_share_doc_weighted": round(nonai_doc, 4),
            "ai_minus_tech_weighted": round(ai_share - nonai_tech, 4),
            "ai_minus_doc_weighted": round(ai_share - nonai_doc, 4),
        })

distinctiveness_df = pd.DataFrame(distinctiveness_records)
distinctiveness_df.to_csv(OUTPUT_DIR / "table2_lens_distinctiveness_by_variant.csv", index=False)

# Pivot for readable comparison
pivot_tech = distinctiveness_df.pivot(index="lens", columns="variant", values="ai_minus_tech_weighted")
pivot_doc = distinctiveness_df.pivot(index="lens", columns="variant", values="ai_minus_doc_weighted")

print("AI minus non-AI lens shares (TECH-WEIGHTED baseline):")
print(pivot_tech.round(3).to_string())
print("\nAI minus non-AI lens shares (DOC-WEIGHTED baseline):")
print(pivot_doc.round(3).to_string())

print("\nIf the rank order of AI-distinctive lenses is similar across variants,")
print("the headline AI-distinctiveness story is robust to prompt variation.")

In [ ]:
# @title Adjusted Rand Index between baseline and each variant
# ARI is computed at the phrase level on phrases that exist in BOTH variants
# for the same paragraph. Where variants produce different numbers of phrases
# per paragraph, we compare the first phrase from each to keep the metric well-
# defined.

def first_phrase_assignment(variant_id):
    sub = phrase_assign_df[phrase_assign_df["variant"] == variant_id]
    sub = sub.sort_values(["chunk_id", "phrase_idx"])
    first = sub.groupby("chunk_id").first().reset_index()
    return first[["chunk_id", "assigned_cluster_id"]].rename(
        columns={"assigned_cluster_id": f"cluster_{variant_id}"}
    )

baseline = first_phrase_assignment("V0")
ari_records = []
for vid in PROMPTS:
    if vid == "V0":
        continue
    variant = first_phrase_assignment(vid)
    joined = baseline.merge(variant, on="chunk_id", how="inner")
    if len(joined) < 10:
        ari = float("nan")
    else:
        ari = adjusted_rand_score(joined["cluster_V0"], joined[f"cluster_{vid}"])
    ari_records.append({
        "variant": vid,
        "n_paragraphs_both_yielded": len(joined),
        "ari_to_v0": round(ari, 3),
    })

ari_df = pd.DataFrame(ari_records)
ari_df.to_csv(OUTPUT_DIR / "table3_ari_to_baseline.csv", index=False)
print(ari_df.to_string(index=False))
print("\nInterpretation: ARI of 1.0 = identical assignments; 0 = random.")
print("ARI > 0.3 in this kind of setting (75 clusters, semantic clustering)")
print("suggests substantial agreement. ARI > 0.5 is strong agreement.")

In [ ]:
# @title Sample disagreement examples for hand-inspection

# Find paragraphs where V0 and V1 (or V0 and V3) assigned phrases to very
# different clusters. Useful for the human to look at when interpreting the
# robustness claim.

EXAMPLES_PER_PAIR = 10

disagreement_rows = []
v0_first = first_phrase_assignment("V0")

for vid in ["V1", "V3"]:
    if vid not in PROMPTS:
        continue
    variant_first = first_phrase_assignment(vid)
    joined = v0_first.merge(variant_first, on="chunk_id", how="inner")
    joined["cluster_V0"] = joined["cluster_V0"].astype(int)
    joined[f"cluster_{vid}"] = joined[f"cluster_{vid}"].astype(int)

    # Get phrases for context
    v0_phrases = (extractions["V0"])
    var_phrases = (extractions[vid])

    disagree = joined[joined["cluster_V0"] != joined[f"cluster_{vid}"]]
    print(f"\n[{vid} vs V0] {len(disagree)}/{len(joined)} paragraphs differ on first-phrase cluster assignment")

    samples = disagree.sample(min(EXAMPLES_PER_PAIR, len(disagree)), random_state=RANDOM_SEED) if len(disagree) > 0 else disagree
    for _, row in samples.iterrows():
        cid = row["chunk_id"]
        para_text = sample_df[sample_df["chunk_id"].astype(str) == str(cid)]["text"].iloc[0]
        v0_p = v0_phrases.get(str(cid), [])
        var_p = var_phrases.get(str(cid), [])
        disagreement_rows.append({
            "comparison": f"V0 vs {vid}",
            "chunk_id": cid,
            "paragraph_text": para_text[:400] + ("..." if len(para_text) > 400 else ""),
            "v0_phrases": " | ".join(v0_p),
            f"{vid}_phrases": " | ".join(var_p),
            "v0_cluster_label": CLUSTER_LABELS.get(int(row["cluster_V0"]), ""),
            f"{vid}_cluster_label": CLUSTER_LABELS.get(int(row[f"cluster_{vid}"]), ""),
        })

if disagreement_rows:
    disagreement_df = pd.DataFrame(disagreement_rows)
    disagreement_df.to_csv(OUTPUT_DIR / "table4_disagreement_examples.csv", index=False)
    print(f"\nWrote {len(disagreement_df)} disagreement examples to table4_disagreement_examples.csv")
    print("Inspect manually to check whether disagreements are substantive or just rewording.")

In [ ]:
# @title Summary table for the paper appendix

# Combine the headline numbers into one short table the paper can use.
summary_rows = []
for vid in PROMPTS:
    diag_row = diag_df[diag_df["variant"] == vid].iloc[0]
    privacy_row = distinctiveness_df[
        (distinctiveness_df["variant"] == vid) &
        (distinctiveness_df["lens"].str.contains("Privacy", case=False, na=False))
    ]
    privacy_diff_tech = privacy_row["ai_minus_tech_weighted"].iloc[0] if len(privacy_row) else np.nan
    privacy_diff_doc = privacy_row["ai_minus_doc_weighted"].iloc[0] if len(privacy_row) else np.nan

    safety_row = distinctiveness_df[
        (distinctiveness_df["variant"] == vid) &
        (distinctiveness_df["lens"].str.contains("Safety", case=False, na=False))
    ]
    safety_diff_tech = safety_row["ai_minus_tech_weighted"].iloc[0] if len(safety_row) else np.nan

    ari_row = ari_df[ari_df["variant"] == vid] if vid != "V0" else None
    ari = ari_row["ari_to_v0"].iloc[0] if ari_row is not None and len(ari_row) else (1.0 if vid == "V0" else np.nan)

    summary_rows.append({
        "variant": vid,
        "name": PROMPTS[vid]["name"],
        "coverage_rate": diag_row["coverage_rate"],
        "phrases_per_paragraph": diag_row["phrases_per_paragraph"],
        "ari_to_baseline": ari,
        "AI_minus_nonAI_Privacy_(tech_w)": privacy_diff_tech,
        "AI_minus_nonAI_Privacy_(doc_w)": privacy_diff_doc,
        "AI_minus_nonAI_Safety_(tech_w)": safety_diff_tech,
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_DIR / "table0_paper_summary.csv", index=False)

print("=== HEADLINE SUMMARY (suitable for paper appendix) ===\n")
print(summary_df.to_string(index=False))

print("\n\nWhat to write in the paper:")
print("- If coverage and phrases-per-paragraph are similar across variants,")
print("  the prompt does not dramatically change WHAT counts as a concern.")
print("- If ARI > ~0.3, baseline and variants assign phrases to similar")
print("  clusters even when the wording differs.")
print("- If 'AI minus non-AI Privacy' stays positive and substantial across")
print("  variants, the AI-distinctiveness story holds under reasonable")
print("  prompt variation.")
print("- Conversely, if any of these dramatically changes between variants,")
print("  that is itself a meaningful finding that should be reported.")

print(f"\nAll outputs written to: {OUTPUT_DIR}")
print("Files:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {p.name}")

## Reading the results

For the paper Methods section, the headline is in `table0_paper_summary.csv`.
The likely text:

> To assess sensitivity to prompt design, we re-ran extraction on a
> stratified random sample of N=500 paragraphs under three prompt variants:
> V0 (baseline, matching the main analysis), V1 ("issues" framing), and
> V3 (strict decontextualisation). Coverage, density, and lens-level
> distinctiveness numbers are reported in Appendix [X].

For the appendix:

- **Table 0** (`table0_paper_summary.csv`): one-line-per-variant summary.
- **Table 1** (`table1_paragraph_diagnostics.csv`): coverage and density.
- **Table 2** (`table2_lens_distinctiveness_by_variant.csv`): full
  lens-level AI-vs-non-AI numbers under each variant.
- **Table 3** (`table3_ari_to_baseline.csv`): cluster-assignment agreement.
- **Table 4** (`table4_disagreement_examples.csv`): disagreement examples
  for hand-inspection.

If headline AI-distinctiveness numbers (Privacy and Safety differences)
hold up across all three variants, write that the conclusion is robust to
prompt variation. If a variant produces qualitatively different numbers,
that is a finding worth reporting in its own right — it would mean the
analysis is sensitive to the analytic category of "concerns" specifically,
which is a substantive claim.

## Costs and timing

Approximate cost on `gpt-5-mini`:
- ~500 paragraphs × 3 variants = 1,500 chat-completion calls.
- ~600 exemplar embeddings + ~3,000 variant phrase embeddings = ~3,600
  embeddings.
- Total cost: typically under $5 USD.
- Wall time: ~25 minutes for extractions + ~5 minutes for embeddings on
  default Colab.
